# Reading From CSV Files

Sources: 
- Bureau of Transportation Statistics - Flights - https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK&QO_fu146_anzr=b0-gvzr
- OurAirports - Airports - https://ourairports.com/data/
- Open Meteo - Historical Weather Data: https://open-meteo.com/en/docs/historical-forecast-api 

In [0]:
# List of file paths and table names

BASE_PATH = "/Volumes/workspace/bronze_travel/travel_dataset"

INGESTION_CONFIG = [
    {
        "source": "bts",
        "path": f"{BASE_PATH}/jan2026.csv",
        "table": "bronze_travel.jan2026"
    },
    {
        "source": "bts",
        "path": f"{BASE_PATH}/feb2026.csv",
        "table": "bronze_travel.feb2026"
    },
    {
        "source": "bts",
        "path": f"{BASE_PATH}/mar2026.csv",
        "table": "bronze_travel.mar2026"
    },
    {
        "source": "bts",
        "path": f"{BASE_PATH}/apr2026.csv",
        "table": "bronze_travel.apr2026"
    },
    {
        "source": "bts",
        "path": f"{BASE_PATH}/may2026.csv",
        "table": "bronze_travel.may2026"
    },
    {
        "source": "bts",
        "path": f"{BASE_PATH}/jun2026.csv",
        "table": "bronze_travel.jun2026"
    },
    {
        "source": "oa",
        "path": f"{BASE_PATH}/airports.csv",
        "table": "bronze_travel.airports"
    },
    {
        "source": "bts",
        "path": f"{BASE_PATH}/airline_carrier.csv",
        "table": "bronze_travel.airlines"
    }
]

# Writing to Bronze Layer

In [0]:
for file in INGESTION_CONFIG:
    print(f"Ingesting {file['path']} into table {file['table']}")

    df = (spark.read.option("header", True)
          .option("inferSchema", True)
          .csv(file["path"]))
    
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(file["table"])

In [0]:
%sql

SELECT *
FROM bronze_travel.jan2026
LIMIT 100

# Gathering Historical Data from Open Meteo
There are over 9000 airports in the world. The flight data is primarily for flights to and from the US. Instead of querying weather data for all airports in the world, query only the relevant airports found in the flight data.

In [0]:
%sql

-- Identifying if there are any duplicate airports

SELECT iata_code, COUNT(*)
FROM (
    SELECT DISTINCT iata_code, longitude_deg, latitude_deg
    FROM bronze_travel.airports
    WHERE iata_code IS NOT NULL
)
GROUP BY iata_code
HAVING COUNT(*) > 1

In [0]:
%sql

-- Identifying iata codes that exist in flight data but not in airport data

WITH airport_list AS (
    SELECT DISTINCT ORIGIN AS iata_code
    FROM bronze_travel.jan2026

    UNION

    SELECT DISTINCT DEST  AS iata_code
    FROM bronze_travel.jan2026

    UNION

    SELECT DISTINCT ORIGIN AS iata_code
    FROM bronze_travel.feb2026

    UNION

    SELECT DISTINCT DEST AS iata_code
    FROM bronze_travel.feb2026

    UNION

    SELECT DISTINCT ORIGIN AS iata_code
    FROM bronze_travel.mar2026

    UNION

    SELECT DISTINCT DEST AS iata_code
    FROM bronze_travel.mar2026

    UNION

    SELECT DISTINCT ORIGIN AS iata_code
    FROM bronze_travel.apr2026

    UNION

    SELECT DISTINCT DEST AS iata_code
    FROM bronze_travel.apr2026

    UNION

    SELECT DISTINCT ORIGIN AS iata_code
    FROM bronze_travel.may2026

    UNION

    SELECT DISTINCT DEST AS iata_code
    FROM bronze_travel.may2026

    UNION

    SELECT DISTINCT ORIGIN AS iata_code
    FROM bronze_travel.jun2026

    UNION

    SELECT DISTINCT DEST AS iata_code
    FROM bronze_travel.jun2026
)

SELECT iata_code
FROM airport_list

EXCEPT

SELECT iata_code
FROM bronze_travel.airports

## Handling airport iata codes that exist in flight data but not in airport csv
- PBI is Palm Beach International Airport and was renamed to DJT on July 9, 2026. The airport csv was updated on September 1, 2026.
    - Use DJT coordinates for PBI iata_code

In [0]:
%sql

SELECT *
FROM bronze_travel.airports
WHERE iata_code = "DJT"

In [0]:
%python

df_airport_codes = spark.sql("""

    WITH airport_list AS (
        SELECT DISTINCT ORIGIN AS iata_code
        FROM bronze_travel.jan2026

        UNION

        SELECT DISTINCT DEST  AS iata_code
        FROM bronze_travel.jan2026

        UNION

        SELECT DISTINCT ORIGIN AS iata_code
        FROM bronze_travel.feb2026

        UNION

        SELECT DISTINCT DEST AS iata_code
        FROM bronze_travel.feb2026

        UNION

        SELECT DISTINCT ORIGIN AS iata_code
        FROM bronze_travel.mar2026

        UNION

        SELECT DISTINCT DEST AS iata_code
        FROM bronze_travel.mar2026

        UNION

        SELECT DISTINCT ORIGIN AS iata_code
        FROM bronze_travel.apr2026

        UNION

        SELECT DISTINCT DEST AS iata_code
        FROM bronze_travel.apr2026

        UNION

        SELECT DISTINCT ORIGIN AS iata_code
        FROM bronze_travel.may2026

        UNION

        SELECT DISTINCT DEST AS iata_code
        FROM bronze_travel.may2026

        UNION

        SELECT DISTINCT ORIGIN AS iata_code
        FROM bronze_travel.jun2026

        UNION

        SELECT DISTINCT DEST AS iata_code
        FROM bronze_travel.jun2026
    )
    SELECT iata_code, longitude_deg, latitude_deg
    FROM bronze_travel.airports
    WHERE iata_code IN (SELECT iata_code FROM airport_list)

    UNION 

    SELECT 
        "PBI" AS iata_code,
        (SELECT longitude_deg FROM bronze_travel.airports WHERE iata_code = "DJT") AS longitude_deg,
        (SELECT latitude_deg FROM bronze_travel.airports WHERE iata_code = "DJT") AS latitude_deg
""")

In [0]:
df_airport_codes.display()

## Open Meteo Rate Limits

There are rate limits for free tier accounts. To avoid rate limit errors, data needs to be retrieved in batches over several minutes. 

Free Tier of Open Meteo has the following rate limits:
- 600 calls / min
- 5,000 calls / hour
- 10,000 calls / day
- 300,000 calls / month

![2026-09-10_18-05-14.png](./2026-09-10_18-05-14.png "2026-09-10_18-05-14.png")


- January 1, 2026 - June 30, 2026 = 181 days
- Querying 8 variables:
    - Min temp (F)
    - Max temp (F)
    - Rain sum (in)
    - Shower sum (in)
    - Snow sum (in)
    - Min wind (mph)
    - Max wind (mph)
- 368 airports = 368 coordinates

Querying 8 variables of weather data for 368 airports for 181 days ~ 3,807 calls. 
3,807 is doable within an hour but only 600 calls are allowed per minute therefore it would take minimum 7 mins or 7 queries/batches.

Below, we will be querying in batch size of 50 which is 8 batches.


In [0]:
import requests
import time
import json
from datetime import datetime

In [0]:
weather_columns = [
    "iata_code",
    "time",
    "temperature_2m_max",
    "temperature_2m_min",
    "rain_sum",
    "showers_sum",
    "snowfall_sum",
    "precipitation_sum",
    "wind_speed_10m_max",
    "wind_gusts_10m_max"
]

In [0]:
# Break up airport list into chunks

def chunk_list(airport_list, batch_size):

    batch_list = []

    for i in range(0, len(airport_list), batch_size):
        batch_list.append(airport_list[i:i + batch_size])

    return batch_list

In [0]:
# Fetch weather data for a single batch of airports

def fetch_weather_data(batch, start_date, end_date):

    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

    latitudes = ",".join(str(airport["latitude_deg"]) for airport in batch)
    longitudes = ",".join(str(airport["longitude_deg"]) for airport in batch)

    params = {
        "latitude": latitudes,
        "longitude": longitudes,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "rain_sum",
            "showers_sum",
            "snowfall_sum",
            "precipitation_sum",
            "wind_speed_10m_max",
            "wind_gusts_10m_max"
        ],
        "timezone": "auto",
        "wind_speed_unit": "mph",
        "temperature_unit": "fahrenheit",
        "precipitation_unit": "inch"
    }

    for attempt in range(3):

        try:

            response = requests.get(url, params=params)

        except requests.exceptions.RequestException as e:

            print(f"Request failed (attempt {attempt + 1}): {e}")
            time.sleep(1)
            continue

        if response.status_code == 200:

            return response.json()

        else:

            print(f"Error {response.status_code}, {response.text}")
            return None

    print("Max retries exceeded for batch, giving up.")
    return None

In [0]:
# Parse weather data for a batch of airports

def parse_weather_data(batch, results):

    weather_rows = []

    for airport, airport_weather in zip(batch, results):

        iata_code = airport["iata_code"]

        daily_data = airport_weather["daily"]

        rows = zip(*daily_data.values())

        for row in rows:

            weather_rows.append((iata_code, *row))

    return weather_rows

In [0]:
# Fetch weather data for all airports

def fetch_all_weather_data(airport_list, batch_size, start_date, end_date):

    batch_list = chunk_list(airport_list, batch_size) 

    weather_results = []

    for batch_no, batch in enumerate(batch_list, start=1):

        print(f"Starting batch {batch_no}")

        batch_weather_data = fetch_weather_data(batch, start_date, end_date)

        weather_rows = parse_weather_data(batch, batch_weather_data)
        
        weather_results.extend(weather_rows)

        print(
        f"Successfully retrieved batch "
        f"{batch_no}/{len(batch_list)}"
        )

        # Sleep for 1 minute to avoid rate limiting
        print(f"Sleeping for 1 minute...")
        time.sleep(60)

    return weather_results

In [0]:
airport_list = df_airport_codes.collect()

In [0]:
airport_list

In [0]:
weather_results = fetch_all_weather_data(airport_list, 50, "2026-01-01", "2026-06-30")
df_weather = spark.createDataFrame(weather_results, weather_columns)

In [0]:
df_weather.display()

In [0]:
# Writing to table

df_weather.write.mode("overwrite").format("delta").saveAsTable("bronze_travel.weather")

In [0]:
%sql

SELECT COUNT(*)
FROM bronze_travel.weather